# Weekly Brief
Fasst die Ergebnisse aller Agenten auf 1-2 Seiten zusammen: **Entscheidung noetig**, **Einlesen**, **kein Handlungsbedarf**.
Die Stufen legt Python mit festen Regeln fest (Thesen-Status, News-Signale, Gewicht vs. Grenze, Termine); das Sprachmodell schreibt nur die Texte.
Voraussetzung: vorher Pipeline bzw. Portfolio Manager laufen lassen (schreibt `research/portfolio_snapshot.json`).

In [ ]:
# 1) Pfade und Imports
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

from finrobot.utils import register_keys_from_json
from lib import brief_agent as ba, pipeline as pl
register_keys_from_json(str(FR / "config_api_keys"))

In [ ]:
# 2) Modell und Titel (Research-Titel; alle Positionen aus dem Portfolio-Snapshot kommen automatisch dazu)
MODEL = "anthropic/claude-haiku-5.5"
llm_config = pl.llm_config_for(MODEL)
TICKERS = ["GOOG", "AMZN", "MA"]

In [ ]:
# 3) Ausloeser ansehen (ohne LLM)
data = ba.collect(TICKERS)
for r in data["positions"]:
    print(f"{r['ticker']:8} {r['level']:7} " + " | ".join(t["text"][:70] for t in r["triggers"][:3]))
print("\nDatenluecken:", *data["gaps"], sep="\n- ")

In [ ]:
# 4) Brief schreiben lassen (1 LLM-Lauf, ca. 1 Minute)
save_dir = f"../reports/brief_{MODEL.split('/')[-1]}"
state = ba.run_brief(llm_config, TICKERS, save_dir, data=data)

In [ ]:
# 5) PDF anzeigen
import io, glob
import pymupdf as fitz
from PIL import Image
pdf = sorted(glob.glob(f"{save_dir}/brief_*.pdf"), key=os.path.getmtime)
for page in (fitz.open(pdf[-1]) if pdf else []):
    display(Image.open(io.BytesIO(page.get_pixmap(dpi=100).tobytes("png"))))

In [ ]:
# 6) HTML-Dashboard (ohne LLM): Startseite mit Action Board und je Report eine Seite mit markierten Zahlen
from lib import html_site
index = html_site.build_site(TICKERS)
print("Im Browser oeffnen:", Path(index).resolve())